In [8]:
import pandas as pd
import evaluate
import re
import os
import numpy as np
# from rouge_score import rouge_scorer, scoring
# from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple, Any, Sequence
# from transformers import AutoModelForCausalLM, AutoTokenizer,AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import math
from neo4j import GraphDatabase, Transaction 

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default


DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS), database=NEO4J_DATABASE)
    

In [9]:
from transformers import RobertaTokenizer, RobertaModel
import torch
from sentence_transformers import SentenceTransformer, util, models


MODEL_NAME = "ehsanaghaei/SecureBERT_Plus"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaModel.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def encode_sentences(texts, batch_size=32, title=""):
    """
    Encode a list of sentences into embeddings using SecureBERT+.
    Uses mean pooling over token embeddings.
    Returns a (len(texts), hidden_size) tensor on CPU.
    """
    if not texts:
        return torch.empty((0, model.config.hidden_size))
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Encoding {title}", unit="batch"):
            batch = texts[i:i + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)

            outputs = model(**enc)
            last_hidden = outputs.last_hidden_state

            # Mean pooling (ignoring padding tokens)
            attention_mask = enc["attention_mask"].unsqueeze(-1)
            masked_hidden = last_hidden * attention_mask
            sum_hidden = masked_hidden.sum(dim=1)
            lengths = attention_mask.sum(dim=1).clamp(min=1)
            embs = sum_hidden / lengths

            all_embs.append(embs.cpu())

    return torch.cat(all_embs, dim=0)

Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT_Plus and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
import pandas as pd

def load_policy_sections(engine) -> pd.DataFrame:
    policy_query = """
    SELECT DISTINCT policy_id, section_id
    , (ARRAY_AGG(section_title ORDER BY row_number))[1] AS section_title 
    , source_framework
    , string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  
    FROM policy_lines
    WHERE policy_origin <> 'template'
    GROUP BY policy_id, section_id, source_framework;
    """
    df = pd.read_sql(policy_query, con=engine).reset_index(drop=True)
    df["section_key"] = df["section_id"].astype(str)
    return df


In [11]:
def load_atomic_requirements(engine) -> pd.DataFrame:
    sql = """
    SELECT 
        subcontrol_atomic_req.id   AS atomic_id,
        subcontrol_atomic_req.subcontrol_id,
        subcontrol_atomic_req.atomic_order,
        subcontrol_atomic_req.atomic_requirement,
        frameworks.name           AS framework_name
    FROM subcontrol_atomic_req
    JOIN subcontrols    ON subcontrol_atomic_req.subcontrol_id = subcontrols.id
    JOIN control_areas  ON subcontrols.control_id = control_areas.id
    JOIN frameworks     ON control_areas.framework_id = frameworks.id
	WHERE subcontrol_atomic_req.is_low_value = False
    """
    df = pd.read_sql(sql, con=engine).reset_index(drop=True)
    return df


In [12]:
from neo4j import GraphDatabase

def get_policy_subcontrol_map(driver) -> pd.DataFrame:
    cypher = """
    MATCH (p:Policy)-[:SATISFIES_REQUIREMENT]->(d:EvidenceRequirement)<-[:REQUIRES_EVIDENCE]-(s:SubControl)
    RETURN DISTINCT
        p.policy_id AS policy_id,
        s.id AS subcontrol_id;
    """
    with driver.session() as session:
        result = session.run(cypher)
        records = [r.data() for r in result]
    if not records:
        return pd.DataFrame(columns=["policy_id", "subcontrol_id"])
    return pd.DataFrame(records)


In [13]:
def build_section_subcontrol_map(policy_sections_df: pd.DataFrame,
                                 policy_subcontrol_df: pd.DataFrame) -> pd.DataFrame:
    # All (policy_id, section_id) pairs
    sec = policy_sections_df[["policy_id", "section_id"]].drop_duplicates()

    section_subcontrol_df = (
        sec.merge(policy_subcontrol_df, on="policy_id", how="left")
    )

    # group to list per section
    section_subcontrol_df = (
        section_subcontrol_df
        .dropna(subset=["subcontrol_id"])
        .groupby(["policy_id", "section_id"])["subcontrol_id"]
        .apply(lambda x: sorted(set(x)))
        .reset_index()
        .rename(columns={"subcontrol_id": "direct_subcontrols"})
    )

    return section_subcontrol_df


In [14]:
policy_sections_df = load_policy_sections(engine)
atomic_df          = load_atomic_requirements(engine)
policy_subcontrol_df = get_policy_subcontrol_map(driver)

section_subcontrol_df = build_section_subcontrol_map(policy_sections_df, policy_subcontrol_df)

policy_sections_df = policy_sections_df.merge(
    section_subcontrol_df,
    on=["policy_id", "section_id"],
    how="left"
)

policy_sections_df["direct_subcontrols"] = policy_sections_df["direct_subcontrols"].apply(
    lambda x: x if isinstance(x, list) else []
)


In [15]:
policy_sections_df

,policy_id,section_id,section_title,source_framework,org_text,section_key,direct_subcontrols
0,CL_0007-AUP-v1.0,CL_0007-AUP-v1.0:S3.7.1,General unacceptable Use,NIST CSF,Privacy violations\n* Dissemination or disclos...,CL_0007-AUP-v1.0:S3.7.1,[9292927.0]
1,CL_0007-ASMP-v1.0,CL_0007-ASMP-v1.0:S5,Audit Controls and Management,NIST CSF,On-demand documented procedures and evidence o...,CL_0007-ASMP-v1.0:S5,"[9292373.0, 9292374.0, 9292376.0, 9292377.0, 9..."
2,CL_0007-ASMP-v1.0,CL_0007-ASMP-v1.0:S4,Policy,NIST CSF,Asset Types\nThe following minimal asset class...,CL_0007-ASMP-v1.0:S4,"[9292373.0, 9292374.0, 9292376.0, 9292377.0, 9..."
3,CL_0007-NSP-v1.1,CL_0007-NSP-v1.1:S2,SCOPE,NIST CSF,This policy applies to all [COMPANY_NAME] pers...,CL_0007-NSP-v1.1:S2,"[9292401.0, 9292415.0, 9292434.0, 9292436.0, 9..."
4,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.2,Asset Inventory,NIST CSF,An information asset inventory shall be establ...,CL_0001-ASMP-v1.0:S3.2,"[9292373.0, 9292374.0, 9292376.0, 9292377.0, 9..."
...,...,...,...,...,...,...,...
1284,CL_0005-CAT-v2.0,CL_0005-CAT-v2.0:S3.2,All-Personnel Training,NIST CSF,All personnel are provided appropriate securit...,CL_0005-CAT-v2.0:S3.2,[]
1285,CL_0007-PVM-v1.2,CL_0007-PVM-v1.2:S3.3,Patching Exceptions,NIST CSF,Patches on production systems (e.g. servers an...,CL_0007-PVM-v1.2:S3.3,"[9292388.0, 9292389.0, 9292428.0, 9292448.0, 9..."
1286,CL_0003-LMP-v1.2,CL_0003-LMP-v1.2:S2,SCOPE,SOC2,This policy applies to all [COMPANY_NAME] pers...,CL_0003-LMP-v1.2:S2,"[9292922.0, 9292925.0, 9292936.0, 9292943.0, 9..."
1287,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,Based upon the job duties and responsibilities...,CL_0001-ASMP-v1.0:S3.10.2,"[9292373.0, 9292374.0, 9292376.0, 9292377.0, 9..."


In [16]:
import torch

# 1) embeddings for atomic requirements
atomic_embeddings = encode_sentences(
    atomic_df["atomic_requirement"].tolist(),
    batch_size=32,
    title="Atomic requirements"
)   # torch.Tensor [n_atomic, dim]

# 2) embeddings for policy sections
section_embeddings = encode_sentences(
    policy_sections_df["org_text"].tolist(),
    batch_size=32,
    title="Policy sections"
)   # torch.Tensor [n_sections, dim]


Encoding Atomic requirements:   0%|          | 0/20 [00:00<?, ?batch/s]

Encoding Policy sections:   0%|          | 0/41 [00:00<?, ?batch/s]

In [17]:
from sentence_transformers import util
import numpy as np

def build_section_atomic_mapping(
    policy_sections_df: pd.DataFrame,
    atomic_df: pd.DataFrame,
    section_embeddings: torch.Tensor,
    atomic_embeddings: torch.Tensor,
    similarity_threshold: float = 0.85,
    top_k: int = 10,
) -> pd.DataFrame:
    rows = []

    # Pre-index atomic rows by framework for quick filtering
    fw_to_atomic_idx = (
        atomic_df
        .reset_index()
        .groupby("framework_name")["index"]
        .apply(list)
        .to_dict()
    )

    for i, sec_row in policy_sections_df.reset_index(drop=True).iterrows():
        fw = sec_row["source_framework"]
        sec_emb = section_embeddings[i]

        # Skip if no atomic reqs for this framework
        if fw not in fw_to_atomic_idx:
            continue

        valid_atomic_idx = fw_to_atomic_idx[fw]
        atomic_subset = atomic_df.loc[valid_atomic_idx].reset_index(drop=False)  # keep original index as 'index'
        atomic_emb_subset = atomic_embeddings[valid_atomic_idx]

        # Similarity: section vs all atomic in SAME framework
        sims = util.pytorch_cos_sim(sec_emb.unsqueeze(0), atomic_emb_subset)[0]  # [n_fw_atomic]

        # Get top_k by similarity
        values, indices = torch.topk(sims, k=min(top_k, sims.shape[0]))
        sim_scores = values.cpu().numpy()
        sim_idx    = indices.cpu().numpy()

        direct_subcontrols = set(sec_row.get("direct_subcontrols", []) or [])

        for j, sim_score in zip(sim_idx, sim_scores):
            atomic_original_idx = atomic_subset.loc[j, "index"]
            a_row = atomic_df.loc[atomic_original_idx]

            sub_id = a_row["subcontrol_id"]
            is_direct = sub_id in direct_subcontrols

            # For inferred, enforce similarity_threshold. For direct, keep even if low.
            if (not is_direct) and (sim_score < similarity_threshold):
                continue

            relation_type = "direct" if is_direct else "inferred"

            rows.append({
                "policy_id": sec_row["policy_id"],
                "section_id": sec_row["section_id"],
                "section_key": sec_row["section_key"],
                "section_title": sec_row["section_title"],
                "source_framework": sec_row["source_framework"],

                "subcontrol_id": sub_id,
                "atomic_id": a_row["atomic_id"],
                "atomic_order": a_row["atomic_order"],
                "atomic_requirement": a_row["atomic_requirement"],

                "similarity": float(sim_score),
                "is_direct": is_direct,
                "relation_type": relation_type,   # "direct" or "inferred"
            })

    return pd.DataFrame(rows)


In [29]:
section_atomic_mapping_df = build_section_atomic_mapping(
    policy_sections_df=policy_sections_df,
    atomic_df=atomic_df,
    section_embeddings=section_embeddings,
    atomic_embeddings=atomic_embeddings,
    similarity_threshold=0.85,  # tweak as needed
    top_k=10                    # tweak as needed
)


In [19]:
out = section_atomic_mapping_df[['policy_id','section_id','section_title','source_framework','subcontrol_id','atomic_id','atomic_order','atomic_requirement','similarity','is_direct','relation_type']]

In [20]:
# out.to_sql('section_atomic_mappings', con=engine, if_exists='replace', index=False)

In [22]:
# Ontology definition table: one row per ontology field (e.g., organization.industry)
ontology_definitions_df = pd.read_sql(
    """
    SELECT
        id,
        label,
        description,
        category,
        seeds,
        counterexamples,
        examples,
        required
    FROM ontology
    """,
    engine,
)

ontology_definitions_df.head()


,id,label,description,category,seeds,counterexamples,examples,required
0,organization.name,Organization Name,The legal or trade name of the organization.,Organization,"{""acme corp"",""company name"",""legal name"",""orga...","{""policy title"",""framework name""}","{""Acme Corporation"",""COMPANY LLC"",""Global FinT...",True
1,organization.industry,Industry,Primary business sector or industry classifica...,Organization,"{healthcare,finance,manufacturing,saas,education}","{""policy area"",""department name""}","{""Healthcare Technology"",""Financial Services"",...",True
2,organization.size,Organization Size,"Approximate company size, typically measured b...",Organization,"{small,mid-size,enterprise,""500 employees""}","{""team size"",""control frequency""}","{""45 employees"",""350 employees"",""1,200 employe...",True
3,organization.regions,Operating Regions,Geographical regions or countries where the or...,Organization,"{""north america"",europe,global,""us only""}","{""data center region"",""policy region""}","{""United States"",""North America and EU"",""Globa...",False
4,organization.tech_stack,Technology Stack,"Core platforms, cloud services, or software pr...",Organization,"{aws,azure,""google workspace"",""microsoft 365""}","{""security tool"",""policy software""}","{""AWS, Microsoft 365, Salesforce"",""Azure + Ser...",False


In [23]:
# Map internal ids (e.g., "organization.industry") to human-friendly labels (e.g., "Industry")
ONTOLOGY_LABELS = (
    ontology_definitions_df
    .set_index("id")["label"]
    .to_dict()
)


In [24]:
def build_concept_text(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("label")):
        parts.append(str(row["label"]).strip())
    if pd.notna(row.get("description")):
        parts.append(str(row["description"]).strip())

    # optional: include examples/seeds to give CyberBERT+ more signal
    ex = str(row.get("examples", "")).strip()
    if ex and ex.lower() != "nan":
        parts.append(f"Examples: {ex}")
    seeds = str(row.get("seeds", "")).strip()
    if seeds and ex.lower() != "nan":
        parts.append(f"Seeds: {seeds}")

    return " ".join(p for p in parts if p)

def prepare_ontology_texts(ontology_df: pd.DataFrame) -> pd.DataFrame:
    df = ontology_df.copy()
    df["concept_text"] = df.apply(build_concept_text, axis=1)
    return df


In [25]:
def build_section_texts(section_atomic_with_onto_df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per (policy_id, section_id), with aggregated atomic text.
    """
    sections_df = (
        section_atomic_with_onto_df
        .groupby(["policy_id", "section_id", "section_title"], as_index=False)
        .agg({
            "atomic_requirement": lambda s: " ".join(
                sorted(set(str(x).strip() for x in s if isinstance(x, str)))
            )
        })
    )
    sections_df["section_text"] = (
        sections_df["section_title"].fillna("").astype(str).str.strip()
        + ". "
        + sections_df["atomic_requirement"].fillna("").astype(str).str.strip()
    )
    return sections_df


def build_subcontrol_texts(section_atomic_with_onto_df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per subcontrol_id, with aggregated atomic text.
    """
    sub_df = (
        section_atomic_with_onto_df
        .groupby(["subcontrol_id"], as_index=False)
        .agg({
            "atomic_requirement": lambda s: " ".join(
                sorted(set(str(x).strip() for x in s if isinstance(x, str)))
            )
        })
    )
    sub_df["subcontrol_text"] = sub_df["atomic_requirement"].fillna("").astype(str).str.strip()
    return sub_df


In [26]:
from typing import Literal
from sentence_transformers import SentenceTransformer, util
import torch

def compute_ontology_alignment(
    ontology_df: pd.DataFrame,
    target_df: pd.DataFrame,
    target_id_cols: list,
    text_col: str,
    model: SentenceTransformer,
    top_k: int = 5,
    min_sim: float = 0.5,
    level: Literal["section", "subcontrol"] = "section",
) -> pd.DataFrame:
    """
    Generic ontology → target (section/subcontrol) mapping.

    ontology_df: must have ['id', 'label', 'concept_text']
    target_df: must contain target_id_cols + [text_col]
    target_id_cols: e.g., ['policy_id','section_id'] or ['subcontrol_id']
    text_col: name of the text column in target_df to embed
    level: "section" or "subcontrol" (just for tagging)
    """
    # Embed ontology concepts
    concept_texts = ontology_df["concept_text"].tolist()
    concept_emb = model.encode(
        concept_texts,
        batch_size=32,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    # Embed targets
    target_texts = target_df[text_col].fillna("").astype(str).tolist()
    target_emb = model.encode(
        target_texts,
        batch_size=32,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    # Similarity matrix: [num_concepts, num_targets]
    cos_sim = util.cos_sim(concept_emb, target_emb)

    align_rows = []

    num_targets = cos_sim.size(1)

    for target_idx in range(num_targets):
        sims = cos_sim[:, target_idx]

        top_vals, top_indices = torch.topk(sims, k=top_k)

        for score, concept_idx in zip(top_vals, top_indices):
            score_val = float(score.item())
            if score_val < min_sim:
                continue

            concept = ontology_df.iloc[int(concept_idx)]
            target = target_df.iloc[int(target_idx)]

            row = {
                "level": level,
                "ontology_id": concept["id"],
                "ontology_label": concept["label"],
                "similarity": score_val,
            }
            # attach target identifiers
            for col in target_id_cols:
                row[col] = target[col]

            align_rows.append(row)

    return pd.DataFrame(align_rows)


In [31]:
# 1) Prepare ontology
ontology_df_prepared = prepare_ontology_texts(ontology_definitions_df)

# 2) Prepare target texts
sections_df = build_section_texts(section_atomic_mapping_df)
subcontrols_df = build_subcontrol_texts(section_atomic_mapping_df)

# 3) Load CyberBERT+ (adjust model name/path)
cb_model = SentenceTransformer("ehsanaghaei/SecureBERT_Plus")  # or your CyberBERT+ variant

# 4) Compute alignments
section_align_df = compute_ontology_alignment(
    ontology_df=ontology_df_prepared,
    target_df=sections_df,
    target_id_cols=["policy_id", "section_id"],
    text_col="section_text",
    model=cb_model,
    top_k=5,
    min_sim=0.5,
    level="section",
)

subcontrol_align_df = compute_ontology_alignment(
    ontology_df=ontology_df_prepared,
    target_df=subcontrols_df,
    target_id_cols=["subcontrol_id"],
    text_col="subcontrol_text",
    model=cb_model,
    top_k=5,
    min_sim=0.5,
    level="subcontrol",
)


No sentence-transformers model found with name ehsanaghaei/SecureBERT_Plus. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT_Plus and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [37]:
ontology_alignment_df = pd.concat([section_align_df, subcontrol_align_df], ignore_index=True)

# Optional: add a run timestamp or model name
from datetime import datetime
ontology_alignment_df["computed_at"] = datetime.utcnow()
ontology_alignment_df["embedding_model"] = "CyberBERT+"

ontology_alignment_df.to_sql("ontology_alignment",engine,if_exists="append",index=False,)

125

In [36]:
ontology_alignment_df

,level,ontology_id,ontology_label,similarity,policy_id,section_id,subcontrol_id,computed_at,embedding_model
0,section,policy.behavior.governance.monitoring_methods,Monitoring & Audit Methods,0.941439,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,NaN,2025-11-25 06:22:20.774435,CyberBERT+
1,section,policy.behavior.scope.assets,Assets in Scope,0.941407,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,NaN,2025-11-25 06:22:20.774435,CyberBERT+
2,section,policy.behavior.behavior.unacceptable_categories,Unacceptable Use Categories,0.938039,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,NaN,2025-11-25 06:22:20.774435,CyberBERT+
3,section,risk.key_concerns,Key Risk Concerns,0.937409,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,NaN,2025-11-25 06:22:20.774435,CyberBERT+
4,section,policy.behavior.controls.network_security,Network & Access Controls,0.936839,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,NaN,2025-11-25 06:22:20.774435,CyberBERT+
...,...,...,...,...,...,...,...,...,...
7120,subcontrol,policy.behavior.governance.monitoring_methods,Monitoring & Audit Methods,0.956538,NaN,NaN,9292970.0,2025-11-25 06:22:20.774435,CyberBERT+
7121,subcontrol,policy.behavior.scope.assets,Assets in Scope,0.950883,NaN,NaN,9292970.0,2025-11-25 06:22:20.774435,CyberBERT+
7122,subcontrol,policy.behavior.controls.network_security,Network & Access Controls,0.949962,NaN,NaN,9292970.0,2025-11-25 06:22:20.774435,CyberBERT+
7123,subcontrol,security.training,Security Awareness Training,0.948569,NaN,NaN,9292970.0,2025-11-25 06:22:20.774435,CyberBERT+
